In [ ]:
import subprocess, sys, platform
import os
import shutil # For removing directories

def _pip(*pkgs, quiet=True):
    cmd = [sys.executable, "-m", "pip", "install"]
    if quiet:
        cmd.append("-q")
    cmd.extend(pkgs)
    result = subprocess.run(cmd, capture_output=True, text=True, check=False)
    # Always print stdout/stderr if not quiet, or if there was an error
    if not quiet or result.returncode != 0:
        print(f"--- pip install command: {' '.join(cmd)} ---")
        print("--- pip stdout ---\n" + result.stdout)
        print("--- pip stderr ---\n" + result.stderr)
    if result.returncode != 0:
        result.check_returncode() # This will raise CalledProcessError

# Update pip and related tools first
_pip("--upgrade", "pip", "setuptools", "wheel", quiet=False)

print("\nAttempting to install numpy, scipy, and pandas first...")
# Explicitly install numpy, scipy, and a compatible pandas version
# This ensures problematic dependencies are satisfied with pre-built wheels for Python 3.13.
_pip("numpy==1.26.4", quiet=False)
_pip("scipy==1.13.0", quiet=False)
_pip("pandas==2.2.3", quiet=False)

# Core scientific stack (excluding pandas, as it's handled above)
_pip("scikit-learn", "matplotlib", "seaborn", "tqdm", "IPython", quiet=False)

print("\nAttempting to install uni2ts's immediate dependencies individually...")
# Install other uni2ts dependencies explicitly, using --no-deps for datasets and gluonts
_pip("einops==0.7.0", quiet=False)
_pip("gluonts~=0.14.3", "--no-deps", quiet=False) # Install gluonts with no-deps to use existing pandas
_pip("hydra-core==1.3.0", quiet=False)
_pip("jaxtyping~=0.2.24", quiet=False)
_pip("lightning>=2.0", quiet=False)
_pip("python-dotenv==1.0.0", quiet=False)
_pip("datasets~=2.17.1", "--no-deps", quiet=False) # Install datasets with no-deps to use existing pandas

print("\nAttempting to install uni2ts from source with --no-deps...")
uni2ts_repo_dir = "uni2ts_repo"
if os.path.exists(uni2ts_repo_dir):
    shutil.rmtree(uni2ts_repo_dir)

# Clone the repository
clone_cmd = ["git", "clone", "https://github.com/SalesforceAIResearch/uni2ts.git", uni2ts_repo_dir]
print(f"--- git clone command: {' '.join(clone_cmd)} ---")
clone_result = subprocess.run(clone_cmd, capture_output=True, text=True, check=False)
print("--- git clone stdout ---\n" + clone_result.stdout)
print("--- git clone stderr ---\n" + clone_result.stderr)
if clone_result.returncode != 0:
    raise subprocess.CalledProcessError(clone_result.returncode, clone_cmd, clone_result.stdout, clone_result.stderr)

# Change directory and install uni2ts
original_cwd = os.getcwd()
os.chdir(uni2ts_repo_dir)
try:
    _pip(".", "--no-deps", quiet=False)
finally:
    os.chdir(original_cwd) # Always change back

# Clean up the cloned repository
if os.path.exists(uni2ts_repo_dir):
    shutil.rmtree(uni2ts_repo_dir)

try:
    import uni2ts
    print("\nSuccessfully imported uni2ts within the installation cell.")
except ImportError as e:
    print(f"\nFailed to import uni2ts within the installation cell: {e}")
    print("This suggests a deeper installation issue, or a need to restart the Colab runtime.")

import torch
print(f"Python  : {platform.python_version()}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()} "
      f"({'GPU: ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'})")

--- pip install command: /usr/bin/python3 -m pip install --upgrade pip setuptools wheel ---
--- pip stdout ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 32.0 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 80.10.2
    Uninstalling setuptools-80.10.2:
      Successfully uninstalled setuptools-80.10.2
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2

--- pip stderr ---
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.


Attempting to install numpy, scipy, and pandas first...
--- 

In [ ]:

# --------------------------------------------------------------
# 2️⃣ Imports & metric helpers (your supplied functions)
# --------------------------------------------------------------
from __future__ import annotations
from dataclasses import dataclass
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
#
# ── Uni2TS imports ────────────────────────────────────────
from uni2ts.dataset import ETTDataset, IllnessDataset   # data loaders
from uni2ts.model.moirai import MoiraiForecaster        # Moirai model
# ── Metric helpers (copy‑pasted from your snippet) ────────
def mse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(mse(y_true, y_pred)))

def mape(y_true: np.ndarray, y_pred: np.ndarray, epsilon: float = 1e-6) -> float:
    denominator = np.maximum(np.abs(y_true), epsilon)
    return float(100.0 * np.mean(np.abs(y_true - y_pred) / denominator))

def temporal_consistency_error(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    if y_true.ndim != 3:
        raise ValueError("Expected [samples, horizon, channels]")
    return float(np.mean(np.abs(np.diff(y_true, axis=1) - np.diff(y_pred, axis=1))))

def _dtw_1d(a: np.ndarray, b: np.ndarray) -> float:
    n, m = len(a), len(b)
    previous = np.full(m + 1, np.inf)
    previous[0] = 0.0
    for i in range(1, n + 1):
        current = np.full(m + 1, np.inf)
        for j in range(1, m + 1):
            current[j] = abs(a[i - 1] - b[j - 1]) + min(
                current[j - 1], previous[j], previous[j - 1]
            )
        previous = current
    return float(previous[m] / max(n + m, 1))

def normalized_dtw(y_true: np.ndarray, y_pred: np.ndarray,
                   max_samples: int = 256, seed: int = 0) -> float:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    rng = np.random.default_rng(seed)
    indices = np.arange(len(y_true))
    if len(indices) > max_samples:
        indices = np.sort(rng.choice(indices, max_samples, replace=False))
    distances = [
        _dtw_1d(y_true[i, :, ch], y_pred[i, :, ch])
        for i in indices for ch in range(y_true.shape[-1])
    ]
    return float(np.mean(distances))

def calibrate_hr_thresholds(training_targets: np.ndarray,
                            training_naive_predictions: np.ndarray,
                            quantiles: tuple[float, ...] = (0.90, 0.95, 0.99)
                            ) -> dict[float, np.ndarray]:
    errors = np.abs(np.asarray(training_targets) - np.asarray(training_naive_predictions))
    return {q: np.quantile(errors, q, axis=0) for q in quantiles}

def hallucination_rate(y_true: np.ndarray, y_pred: np.ndarray,
                       thresholds: np.ndarray) -> float:
    violations = np.abs(np.asarray(y_true) - np.asarray(y_pred)) > np.asarray(thresholds)
    return float(np.mean(violations))

def range_violation_rate(y_pred: np.ndarray,
                         training_minimum: np.ndarray,
                         training_maximum: np.ndarray) -> float:
    y_pred = np.asarray(y_pred)
    return float(np.mean((y_pred < training_minimum) | (y_pred > training_maximum)))

@dataclass
class GaussianCalibration:
    bias: np.ndarray
    sigma: np.ndarray

def fit_gaussian_calibration(y_true: np.ndarray, y_pred: np.ndarray) -> GaussianCalibration:
    residual = np.asarray(y_true) - np.asarray(y_pred)
    return GaussianCalibration(
        bias=np.mean(residual, axis=0),
        sigma=np.maximum(np.std(residual, axis=0), 1e-6),
    )

def gaussian_nll(y_true: np.ndarray, y_pred: np.ndarray,
                 calibration: GaussianCalibration) -> float:
    residual = np.asarray(y_true) - np.asarray(y_pred) - calibration.bias
    variance = calibration.sigma**2
    value = 0.5 * (np.log(2 * np.pi * variance) + residual**2 / variance)
    return float(np.mean(value))

def evaluate_forecasts(y_true: np.ndarray, y_pred: np.ndarray,
                       hr_thresholds: dict[float, np.ndarray],
                       training_minimum: np.ndarray,
                       training_maximum: np.ndarray,
                       gaussian_calibration: GaussianCalibration | None = None,
                       mape_epsilon: float = 1e-6,
                       dtw_max_samples: int = 256,
                       dtw_seed: int = 0) -> dict[str, float]:
    results = {
        "mse": mse(y_true, y_pred),
        "mae": mae(y_true, y_pred),
        "rmse": rmse(y_true, y_pred),
        "mape": mape(y_true, y_pred, mape_epsilon),
        "tce": temporal_consistency_error(y_true, y_pred),
        "dtw": normalized_dtw(y_true, y_pred, dtw_max_samples, dtw_seed),
        "range_violation_rate": range_violation_rate(
            y_pred, training_minimum, training_maximum
        ),
    }
    for quantile, thresholds in hr_thresholds.items():
        results[f"hr_q{int(round(quantile * 100)):02d}"] = hallucination_rate(
            y_true, y_pred, thresholds
        )
    if gaussian_calibration is not None:
        results["gaussian_nll"] = gaussian_nll(y_true, y_pred, gaussian_calibration)
    return results


ModuleNotFoundError: No module named 'uni2ts.dataset'

In [ ]:
# --------------------------------------------------------------
# 3️⃣ Helper: build train/val/test splits for any dataset
# --------------------------------------------------------------
def make_splits(series: np.ndarray,
                input_len: int,
                horizon: int,
                val_ratio: float = 0.1,
                test_ratio: float = 0.1) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Convert a 1‑D (or 2‑D) time‑series into overlapping windows.
    Returns (train, val, test), each shaped [samples, horizon, channels].
    """
    n = series.shape[0]
    total_len = input_len + horizon
    windows = np.lib.stride_tricks.sliding_window_view(series, (total_len, series.shape[-1]))
    windows = windows.squeeze(0)                     # [samples, total_len, channels]
    X = windows[:, :input_len, :]                    # not used directly by Moirai, but kept for completeness
    Y = windows[:, input_len:, :]                    # forecast target

    # split indices
    n_samples = Y.shape[0]
    test_start = int(n_samples * (1 - test_ratio))
    val_start = int(n_samples * (1 - test_ratio - val_ratio))
    train = Y[:val_start]
    val   = Y[val_start:test_start]
    test  = Y[test_start:]
    return train, val, test


In [ ]:
# --------------------------------------------------------------
# 4️⃣ Configuration – datasets, horizons, and model hyper‑params
# --------------------------------------------------------------
# Datasets we will use
DATASETS = {
    "ETT1":  ETTDataset(root=".", name="ETT1", freq="h"),
    #"ETTh2": ETTDataset(root=".", name="ETTh2", freq="h"),
    #"ETTm1": ETTDataset(root=".", name="ETTm1", freq="t"),
    #"ETTm2": ETTDataset(root=".", name="ETTm2", freq="t"),
    #"Illness": IllnessDataset(root=".", name="Illness", freq="h"),
}

# Forecast horizons (in steps)
HORIZONS = {
    "ETT":  [96, 192, 336, 720],
    #"Illness": [24, 36, 48, 60],
}

# Fixed input length (Moirai paper uses 96 by default)
INPUT_LEN = 96

# Moirai model hyper‑parameters (same for all runs – feel free to tweak)
MOIRAI_CFG = {
    "model_dim": 64,
    "num_blocks": 2,
    "num_layers": 2,
    "dropout": 0.1,
    "batch_size": 64,
    "max_epochs": 30,
    "learning_rate": 1e-3,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}


NameError: name 'ETTDataset' is not defined

In [ ]:
# --------------------------------------------------------------
# 5️⃣ Core loop – train + evaluate for each (dataset, horizon)
# --------------------------------------------------------------
results_report = []

for ds_name, ds in DATASETS.items():
    print(f"\n=== Dataset: {ds_name} ===")
    # Load raw series as np.ndarray (shape: [time, channels])
    series = ds.load().astype(np.float32)       # Uni2TS returns a pandas.DataFrame; convert to np
    series = series.values                       # [T, C]

    # Choose horizon list based on dataset type
    horizons = HORIZONS["Illness"] if ds_name == "Illness" else HORIZONS["ETT"]

    for horizon in horizons:
        print(f"\n-- Horizon = {horizon} steps --")
        # ---------------------------------------------------------
        # Build train/val/test windows
        # ---------------------------------------------------------
        train_y, val_y, test_y = make_splits(series, INPUT_LEN, horizon)

        # Fit simple “naive” baseline (last observed value) – needed for HR calibration
        naive_pred = np.repeat(train_y[:, -1:, :], horizon, axis=1)  # shape matches train_y
        hr_thresholds = calibrate_hr_thresholds(train_y, naive_pred)

        # Compute range statistics for the *training* targets
        training_min = train_y.min(axis=(0, 1), keepdims=True)
        training_max = train_y.max(axis=(0, 1), keepdims=True)

        # ---------------------------------------------------------
        # Build & train Moirai forecaster
        # ---------------------------------------------------------
        forecaster = MoiraiForecaster(
            input_size=series.shape[-1],
            input_length=INPUT_LEN,
            output_length=horizon,
            **MOIRAI_CFG,
        )

        # Convert numpy windows to torch tensors (required by Moirai)
        train_tensor = torch.from_numpy(train_y).float()
        val_tensor   = torch.from_numpy(val_y).float()

        forecaster.fit(
            train_loader=torch.utils.data.DataLoader(
                torch.utils.data.TensorDataset(train_tensor),
                batch_size=MOIRAI_CFG["batch_size"],
                shuffle=True,
            ),
            val_loader=torch.utils.data.DataLoader(
                torch.utils.data.TensorDataset(val_tensor),
                batch_size=MOIRAI_CFG["batch_size"],
                shuffle=False,
            ),
            verbose=True,
        )

        # ---------------------------------------------------------
        # Inference on test set
        # ---------------------------------------------------------
        forecaster.eval()
        test_tensor = torch.from_numpy(test_y).float()
        test_loader = torch.utils.data.DataLoader(
            torch.utils.data.TensorDataset(test_tensor),
            batch_size=MOIRAI_CFG["batch_size"],
            shuffle=False,
        )

        all_preds = []
        with torch.no_grad():
            for (batch,) in test_loader:
                pred = forecaster.predict(batch)          # shape [B, horizon, C]
                all_preds.append(pred.cpu().numpy())
        y_pred = np.concatenate(all_preds, axis=0)          # [N_test, horizon, C]

        # ---------------------------------------------------------
        # (Optional) Gaussian calibration on validation set
        # ---------------------------------------------------------
        # Fit Gaussian calibration on validation forecasts (you can skip if you prefer)
        val_pred = forecaster.predict(val_tensor).cpu().numpy()
        gauss_cal = fit_gaussian_calibration(val_y, val_pred)

        # ---------------------------------------------------------
        # Evaluate with the full metric suite you provided
        # ---------------------------------------------------------
        scores = evaluate_forecasts(
            y_true=test_y,
            y_pred=y_pred,
            hr_thresholds=hr_thresholds,
            training_minimum=training_min,
            training_maximum=training_max,
            gaussian_calibration=gauss_cal,
        )
        # Store results for a tidy final table
        summary = {"dataset": ds_name, "horizon": horizon, **scores}
        results_report.append(summary)

        # Show a quick view for this horizon
        print(pd.Series(scores).sort_values().head(10))


In [ ]:
# --------------------------------------------------------------
# 6️⃣ Aggregate all results into a pretty DataFrame
# --------------------------------------------------------------
import pandas as pd
df = pd.DataFrame(results_report)
df = df[["dataset", "horizon",
         "mse", "mae", "rmse", "mape",
         "tce", "dtw", "range_violation_rate",
         "hr_q90", "hr_q95", "hr_q99",
         "gaussian_nll"]]
df.sort_values(["dataset", "horizon"], inplace=True)
display(df)


In [ ]:
# --------------------------------------------------------------
# 7️⃣ (Optional) Save the results to CSV for later analysis
# --------------------------------------------------------------
df.to_csv("moirai_forecasting_results.csv", index=False)
print("✅ Results saved to `moirai_forecasting_results.csv`")
